# Pair Export and Analysis

Assembles the prediction log into comparisons and reports the result of each.

This notebook reports the constructed set. Unaltered Bridge frames remain in the
prediction log for a later validation analysis, once those scenes have been
reviewed; they are not mixed into the numbers below.

The analysis is ordered so that later results are read in the light of earlier
ones. The instrument checks come first: if the lateral output does not respond to
the scene, or if the measurement sits below the quantisation floor, then nothing
downstream can be interpreted, and reporting a language result without them would
be reporting an artefact. The experiments follow, then the controls.

The constructed scenes are the frozen set from Notebook 03: screened for stimulus
quality and selected in paired form, with each same-side scene accompanied by the
opposite scene built from the same base frame. That pairing is what makes the
decisive contrast a within-frame comparison.

Three conventions apply throughout.

Statistics are computed on the continuous expected-bin readout rather than on the
argmax action, because the argmax is quantised at roughly the size of the effect
being measured. Coverage is reported rather than assumed.

Each comparison is read on the axis its spatial term contrasts, taken from
`axis_index`. The earlier analysis read the lateral component for every pair,
including front/back and top/bottom terms that have no reason to move it, which
diluted the measurement with rows that could not have shown an effect.

A null is reported with an equivalence test against one action bin width, not
with a non-significant p value. Where the finding is that the model does not
respond, the claim is that the effect is negligible, and that is a claim a
difference test cannot make.

## 1. Mount Drive

In [31]:
from google.colab import drive
drive.mount('/content/drive')
import os
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/v2/bridge'
CONSTRUCTED_DIR = '/content/drive/MyDrive/openvla_cache/v2/constructed'
PROBE_CSV = '/content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv'
PAIRS_JSON = '/content/drive/MyDrive/openvla_cache/v2/pairs.json'
MANIFEST = os.path.join(CACHE_DIR, 'manifest.csv')
print('log      ->', PROBE_CSV)
print('manifest ->', MANIFEST)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
log      -> /content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv
manifest -> /content/drive/MyDrive/openvla_cache/v2/bridge/manifest.csv


## 2. Import the code

Clones the project code from GitHub into the runtime, so the analysis functions
always match the pushed commit. No OpenVLA load is needed.

In [32]:
import sys, os, importlib, subprocess

REPO_URL = 'https://github.com/LewisTL/ECS8056.git'
BRANCH = 'master'
REPO_DIR = '/content/ECS8056'

def sync_repo():
    """Clone or hard-refresh the repository so it matches origin/BRANCH."""
    token = os.environ.get('GITHUB_TOKEN', '')
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', url],
                       check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--depth', '1',
                        'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', '--quiet',
                        f'origin/{BRANCH}'], check=True)
    else:
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch',
                        BRANCH, url, REPO_DIR], check=True)
    return subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
                          capture_output=True, text=True).stdout.strip()


commit = sync_repo()
module_dir = REPO_DIR
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
for _m in ('action_bins', 'prediction_log', 'data', 'controls', 'compose_scenes', 'export_pairs', 'analysis',):
    sys.modules.pop(_m, None)
importlib.invalidate_caches()

import analysis
from analysis import (axis_value, wilcoxon_paired, tost_equivalence,
                      resolution_report, mirror_check,
                      mirror_check_by_configuration, lexical_check,
                      term_effect, congruence_test, absolute_congruence,
                      same_side_test, CONTINUOUS_COLS)
from export_pairs import load_inputs, build_pairs, write_pairs
from compose_scenes import (evaluation_scenes, load_constructed_manifest,
                            manipulation_rate, IMAGE_X_TO_LATERAL_SIGN)
from data import AXIS_INDEX
print(f'imported project modules from {module_dir} @ {commit}')

imported project modules from /content/ECS8056 @ 41be4c2


## 3. Load the log and report coverage

Every result below is conditional on what was actually collected, so the
composition of the log is printed before any statistic. Unequal counts per
condition are expected: a condition is skipped when its precondition fails, for
example a mirror condition on a depth term, and skipping keeps each logged
prediction interpretable at the cost of a balanced design.

Unaltered Bridge rows are counted, then held back, so the coverage that follows
describes the constructed set alone.

In [33]:
import numpy as np
import pandas as pd

log = pd.read_csv(PROBE_CSV)
n_bridge = int((log['scene_source'] == 'bridge').sum())
print(f'{len(log)} predictions in the log')
print(f'{n_bridge} unaltered Bridge rows held back')

coverage = log[CONTINUOUS_COLS].notna().all(axis=1)
constructed = log['scene_source'] == 'constructed'
usable = log[coverage & constructed].copy()
print(f'\ncontinuous readout present on {coverage.sum()}/{len(log)} rows '
      f'({coverage.mean():.1%})')
if coverage.mean() < 1.0:
    print('rows without it are excluded from the continuous comparisons; the log '
          'for this generation is written by the continuous path throughout, so a '
          'gap here means an interrupted or hand-edited file rather than a '
          'migrated row')

# Every comparison below reads the continuous value on the row's own axis.
usable['value'] = axis_value(usable)
print(f'\nconstructed predictions used: {len(usable)}')
print('\nby condition:')
print(usable['condition'].value_counts().sort_index().to_string())
print('\nby axis:', dict(usable['axis'].value_counts()))

10196 predictions in the log
8276 unaltered Bridge rows held back

continuous readout present on 10196/10196 rows (100.0%)

constructed predictions used: 1920

by condition:
condition
baseline          480
mirror            480
mirror_neutral    240
neutral           240
swapped_scene     480

by axis: {'lateral': np.int64(1920)}


/tmp/ipykernel_468/3990646217.py:4: DtypeWarning: Columns (56) have mixed types. Specify dtype option on import or set low_memory=False.
  log = pd.read_csv(PROBE_CSV)


## 4. Export pairs

Groups the constructed rows of the log into comparisons by
(pair_id, frame, condition, scene_source) and writes `pairs.json` for the Isaac
Sim visualiser. Unaltered Bridge predictions are held back, matching the analysis
below.

Scene labels are re-joined from the manifest rather than read from the log. The
log freezes whatever a label was when the probe ran, which previously hid every
label added by review afterwards, and is why the manifest showed 15 pairable
feasible referent scenes while the analysis found 7.

In [34]:
preds, manifest = load_inputs(PROBE_CSV, MANIFEST)
n_bridge_preds = int((preds['scene_source'] == 'bridge').sum())
preds = preds[preds['scene_source'] == 'constructed']
print(f'held back {n_bridge_preds} unaltered Bridge predictions from the export')
pairs, stats = build_pairs(preds, manifest)
print(f"{stats['paired']} paired comparisons, {stats['neutral']} single-role records")
if stats['incomplete']:
    print(f"{stats['incomplete']} groups are missing a role the condition calls "
          "for; the probe run was most likely interrupted")
write_pairs(pairs, PAIRS_JSON)

/content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv
  67 columns, 10196 rows, consistent


/content/ECS8056/export_pairs.py:158: DtypeWarning: Columns (56) have mixed types. Specify dtype option on import or set low_memory=False.
  preds = pd.read_csv(pred_path)


held back 8276 unaltered Bridge predictions from the export
720 paired comparisons, 480 single-role records
[write_pairs] wrote 1200 pairs -> /content/drive/MyDrive/openvla_cache/v2/pairs.json


'/content/drive/MyDrive/openvla_cache/v2/pairs.json'

## 5. Instrument check: is the lateral channel live?

With the instruction held fixed, mirroring the image reverses the lateral axis of
the scene, so a model that reads lateral position must change the sign of its
lateral output. The term-stripped variant isolates object grounding from any
influence of the spatial word.

This is the gate. If the response does not move when the whole scene is
reflected, then the visual channel is not live on this axis, and a null on the
language comparisons follows from the model ignoring the image rather than from
anything about spatial language. It also identifies the lateral axis empirically,
which the ground-truth pilot could not do because the mapping between BridgeData
action axes and the camera frame was never resolved.

Passing this check is a necessary condition for the language claims, not evidence
for them: reaching toward the only object in view requires no spatial language at
all.

The constructed scenes are reported per arrangement rather than pooled, because
the arrangements are not interchangeable here. Reflecting an `opposite`
arrangement maps the layout onto itself, so a response near the midpoint produces
an antisymmetry and an invariance both near zero and neither decides anything; the
same-side arrangements are where a reflection genuinely moves the scene. Pooling
them would dilute the only stratum that can pass or fail the check.

In [35]:
def print_mirror(result, label):
    print(f'--- {label}')
    for kind in ('neutral', 'term'):
        entry = result.get(kind, {})
        if not entry.get('n'):
            print(f'  {kind}: no paired mirror predictions')
            continue
        print(f"  {kind:8} n={entry['n']:4}  flips={entry['flip_rate']:.1%}  "
              f"identical={entry['identical_rate']:.1%}  "
              f"mean |lateral|={entry['mean_abs_original']:.5f}  "
              f"mean |change|={entry['mean_abs_change']:.5f}")
        print(f"           antisymmetry median={entry['antisymmetry']['median']:+.5f} "
              f"p={entry['antisymmetry']['p_value']:.3g}   "
              f"invariance median={entry['invariance']['median']:+.5f} "
              f"p={entry['invariance']['p_value']:.3g}")
    return result

def print_mirror(result, label):
    print(f'--- {label}')
    for kind in ('neutral', 'term'):
        entry = result.get(kind, {})
        if not entry.get('n'):
            print(f'  {kind}: no paired mirror predictions')
            continue
        print(f"  {kind:8} n={entry['n']:4}  flips={entry['flip_rate']:.1%}  "
              f"identical={entry['identical_rate']:.1%}  "
              f"mean |lateral|={entry['mean_abs_original']:.5f}  "
              f"mean |change|={entry['mean_abs_change']:.5f}")
        print(f"           antisymmetry median={entry['antisymmetry']['median']:+.5f} "
              f"p={entry['antisymmetry']['p_value']:.3g}   "
              f"invariance median={entry['invariance']['median']:+.5f} "
              f"p={entry['invariance']['p_value']:.3g}")
    return result

lateral = usable[usable['axis_index'] == AXIS_INDEX['lateral']]

# Per arrangement. The same-side entries are the ones that can establish the
# necessary condition; the opposite entry is reported so the comparison between
# them is visible rather than assumed.
mirror_built = mirror_check_by_configuration(lateral)
for configuration, result in sorted(mirror_built.items()):
    print_mirror(result, configuration or 'unlabelled')

# The same-side arrangements pooled, which is the figure the summary reads: both
# are informative, and the side an arrangement was built on is not a factor the
# check is about.
mirror_built_same_side = mirror_check(
    lateral[lateral['configuration'].astype(str).str.startswith('same_side')])

print('\nRead: a high flip rate with an antisymmetry median near zero means the '
      'lateral channel tracks the scene. A near-zero flip rate with an invariance '
      'median near zero means the output ignores the image, and no language '
      'conclusion can be drawn from this axis. Read the same-side arrangements '
      'for the verdict: the opposite arrangement is close to symmetric under '
      'reflection, so a small change there is expected.')

# The same-side arrangements pooled, which is the figure the summary reads: both
# are informative, and the side an arrangement was built on is not a factor the
# check is about.
mirror_built_same_side = mirror_check(
    lateral[(lateral['scene_source'] == 'constructed')
            & lateral['configuration'].astype(str).str.startswith('same_side')])

print('\nRead: a high flip rate with an antisymmetry median near zero means the '
      'lateral channel tracks the scene. A near-zero flip rate with an invariance '
      'median near zero means the output ignores the image, and no language '
      'conclusion can be drawn from this axis. Read the same-side arrangements '
      'for the verdict: the opposite arrangement is close to symmetric under '
      'reflection, so a small change there is expected.')

--- opposite
  neutral  n= 120  flips=22.5%  identical=0.0%  mean |lateral|=0.00448  mean |change|=0.00463
           antisymmetry median=-0.00547 p=5.22e-09   invariance median=-0.00083 p=0.0332
  term     n= 120  flips=25.8%  identical=0.0%  mean |lateral|=0.00422  mean |change|=0.00457
           antisymmetry median=-0.00501 p=1.09e-07   invariance median=-0.00021 p=0.462
--- same_side_left
  neutral  n=  64  flips=32.8%  identical=0.0%  mean |lateral|=0.00507  mean |change|=0.00461
           antisymmetry median=-0.00503 p=7.11e-05   invariance median=-0.00211 p=0.00012
  term     n=  64  flips=25.0%  identical=0.0%  mean |lateral|=0.00447  mean |change|=0.00393
           antisymmetry median=-0.00495 p=1.34e-05   invariance median=-0.00131 p=0.0037
--- same_side_right
  neutral  n=  56  flips=25.0%  identical=0.0%  mean |lateral|=0.00394  mean |change|=0.00559
           antisymmetry median=-0.00544 p=4.08e-05   invariance median=+0.00172 p=0.000527
  term     n=  56  flips=25.0% 

## 6. Measurement resolution

How much of each comparison survives the action quantisation. A distribution
dominated by exact zeros, or by differences smaller than one bin, is at the
resolution floor of the argmax readout, and no conclusion drawn from its sign is
safe.

This is reported as a first-class result rather than a footnote, because it is
the defect that made the earlier findings uninterpretable: the median paired
difference was exactly zero in every stratum, and five of eight sampled pairs
were bit-identical between the two instructions. The comparison between the
argmax and continuous columns here is the direct evidence of what the continuous
readout recovered.

In [36]:
# Set from the model's own decoding constants, printed by Notebook 05. The
# lateral entry of describe_action_space()['bin_width'].
BIN_WIDTH = 0.000225

def paired_difference(frame, condition='baseline', continuous=True):
    """Signed A minus B difference per scene on that scene's own axis."""
    sub = frame[frame['condition'] == condition].copy()
    if sub.empty:
        return pd.DataFrame(columns=['scene_id', 'diff'])
    sub['value'] = axis_value(sub, continuous=continuous)
    wide = sub.pivot_table(index='scene_id', columns='role', values='value')
    if not {'a', 'b'} <= set(wide.columns):
        return pd.DataFrame(columns=['scene_id', 'diff'])
    wide = wide.dropna(subset=['a', 'b'])
    return pd.DataFrame({'scene_id': wide.index, 'diff': (wide['a'] - wide['b']).to_numpy()})

rows = []
for source in sorted(usable['scene_source'].unique()):
    frame = usable[usable['scene_source'] == source]
    for readout, flag in (('argmax', False), ('continuous', True)):
        diffs = paired_difference(frame, continuous=flag)['diff']
        report = resolution_report(diffs, BIN_WIDTH)
        rows.append({'scene_source': source, 'readout': readout, **report})

resolution = pd.DataFrame(rows)
print(resolution[['scene_source', 'readout', 'n', 'frac_exact_zero',
                  'frac_below_bin', 'n_distinct']].to_string(index=False))
print(f'\nbin width used: {BIN_WIDTH:.7f}')
print('A continuous row with far fewer exact zeros than its argmax row is the '
      'measurement the quantisation was hiding.')

scene_source    readout   n  frac_exact_zero  frac_below_bin  n_distinct
 constructed     argmax 240         0.408333        0.450000         123
 constructed continuous 240         0.000000        0.279167         231

bin width used: 0.0002250
A continuous row with far fewer exact zeros than its argmax row is the measurement the quantisation was hiding.


## 7. The experiments: constructed scenes

### 7a. Direction of the paired difference

The expected direction comes from where the two instances were placed, not from
the wording of the instruction.

This is a direction check rather than the decisive test, and the distinction is
worth being explicit about. In the `opposite` configuration the term's
conventional direction and the actual layout coincide by construction, so a model
that maps `left` to a fixed direction without ever consulting the scene agrees
here just as completely as a grounded one. What the statistic establishes is that
the difference is oriented consistently with the geometry at all, which is a
precondition for reading its sign, and it fixes the convention relating image
position to the lateral action empirically. Sections 7b and 7c carry the actual
discrimination.

In [37]:
built = usable[usable['scene_source'] == 'constructed']
if built.empty:
    print('no constructed predictions in the log; run Notebooks 03 and 05 first')
else:
    # The recorded geometry is logged in image coordinates, so the convention
    # relating image position to the sign of the lateral action is applied here.
    congruence = congruence_test(built, lateral_sign=IMAGE_X_TO_LATERAL_SIGN)
    print(f"n={congruence['n']}  agreement with geometry={congruence['agreement']:.1%}")
    test = congruence['test']
    print(f"  median oriented difference={test['median']:+.6f}  "
          f"p={test['p_value']:.3g}  effect size={test['rank_biserial']:+.2f}")
    print('\nby configuration:')
    for name, entry in sorted(congruence['by_configuration'].items()):
        print(f"  {name:16} n={entry['n']:4}  agreement={entry['agreement']:.1%}  "
              f"p={entry['test']['p_value']:.3g}  "
              f"effect={entry['test']['rank_biserial']:+.2f}")
    print(f'\nsign convention in use: IMAGE_X_TO_LATERAL_SIGN = '
          f'{IMAGE_X_TO_LATERAL_SIGN:+d}')
    print('Agreement well below one half would mean the convention relating '
          'image position to the lateral action is inverted. Since the log holds '
          'the geometry in image coordinates, flipping the constant in '
          'compose_scenes.py and re-running this notebook is sufficient; no '
          'prediction needs to be made again.')

n=240  agreement with geometry=46.7%
  median oriented difference=-0.000026  p=0.117  effect size=-0.12

by configuration:
  opposite         n= 120  agreement=39.2%  p=0.00875  effect=-0.28
  same_side_left   n=  64  agreement=53.1%  p=0.574  effect=-0.08
  same_side_right  n=  56  agreement=55.4%  p=0.138  effect=+0.23

sign convention in use: IMAGE_X_TO_LATERAL_SIGN = +1
Agreement well below one half would mean the convention relating image position to the lateral action is inverted. Since the log holds the geometry in image coordinates, flipping the constant in compose_scenes.py and re-running this notebook is sufficient; no prediction needs to be made again.


### 7b. The decisive comparison: same side versus opposite

On a scene with both instances on the same side of the start position, scene
grounding and a word-to-direction mapping predict opposite things, and nothing
else separates them. A grounded model selects between two targets that lie in the
same direction, so both instructions produce a same-signed action differing only
in magnitude. A model that maps `left` to one direction and `right` to the other
produces oppositely-signed actions, since the arrangement plays no part.

On the `opposite` configuration the two accounts agree. That is exactly why the
original sign-flip metric could not distinguish them: it scored the behaviour
they share, so a perfect score was consistent with the model never having looked
at the scene. The contrast between configurations is the quantity that
discriminates, and it exists only because the same-side arrangement was
constructed rather than sought.

Two properties of how it is reported here follow from the construction. The
contrast is computed paired within the base frame as well as across all scenes,
because the frozen set builds both arrangements from the same frame: the paired
form holds the background, the object, the cutout, and the instruction fixed and
leaves the arrangement as the only difference, which the unpaired form cannot
claim. And it is computed twice, once counting every nonzero prediction and once
counting only predictions of at least one action bin, since a sub-bin sign is not
a decision the model could execute and counting it as one drags every rate toward
one half.

In [38]:
if not built.empty:
    # Reported twice: once counting every nonzero prediction, and once counting
    # only predictions of at least one action bin. A sub-bin sign is not a
    # decision the model could execute, and treating it as one pulls every rate
    # toward one half, which attenuates exactly this contrast.
    for label, floor in (('every nonzero prediction', 0.0),
                         (f'at least one bin ({BIN_WIDTH:.6f})', BIN_WIDTH)):
        same_side = same_side_test(built, min_magnitude=floor)
        print(f'--- same-signed action rate, {label}')
        for name, entry in sorted(same_side['by_configuration'].items()):
            print(f"  {name:16} n={entry['n']:4}  "
                  f"same sign={entry['same_sign_rate']:.1%}  "
                  f"undecided={entry['n_unresolved']}  "
                  f"median |magnitude gap|={entry['median_magnitude_gap']:.6f}")
        contrast = same_side['contrast']
        print(f"  contrast (same side minus opposite): "
              f"{contrast['difference']:+.1%}  p={contrast['p_value']:.3g}")
        paired = same_side['contrast_paired']
        print(f"  paired within the base frame: {paired['difference']:+.1%}  "
              f"pairs={paired['n_pairs']}  discordant={paired['n_discordant']}  "
              f"p={paired['p_value']:.3g}  unpaired scenes={paired['n_unpaired']}")

    print('\nRead: a large positive contrast is scene grounding. A contrast near '
          'zero, with both configurations flipping sign, is a word-to-direction '
          'mapping that never consulted the arrangement. The paired figure is the '
          'one to quote: it compares the two arrangements built from the same '
          'frame, so background, object, cutout, and instruction are all held '
          'fixed and the arrangement is the only difference.')

--- same-signed action rate, every nonzero prediction
  opposite         n= 120  same sign=88.3%  undecided=0  median |magnitude gap|=0.000750
  same_side_left   n=  64  same sign=84.4%  undecided=0  median |magnitude gap|=0.000581
  same_side_right  n=  56  same sign=83.9%  undecided=0  median |magnitude gap|=0.000501
  contrast (same side minus opposite): -4.2%  p=0.454
  paired within the base frame: -4.2%  pairs=120  discordant=23  p=0.405  unpaired scenes=0
--- same-signed action rate, at least one bin (0.000225)
  opposite         n= 111  same sign=88.3%  undecided=9  median |magnitude gap|=0.000861
  same_side_left   n=  59  same sign=89.8%  undecided=5  median |magnitude gap|=0.000547
  same_side_right  n=  54  same sign=85.2%  undecided=2  median |magnitude gap|=0.000447
  contrast (same side minus opposite): -0.7%  p=1
  paired within the base frame: +1.9%  pairs=107  discordant=16  p=0.804  unpaired scenes=10

Read: a large positive contrast is scene grounding. A contrast ne

### 7c. Does each instruction move toward its own target?

The same discrimination stated directly. Rather than scoring the pair against
the relative order of the two targets, each instruction is scored against the
side its own target occupies.

On a same-side scene both targets lie in one direction, so a grounded model
agrees on both instructions, while a word-to-direction mapping necessarily
disagrees on one: it sends `left` one way and `right` the other, and only one of
those can point at a target when both targets share a side. The expected
agreement is therefore near one for a grounded model and near one half for a
lexical one. The paired difference cannot show this, because the two accounts
produce the same difference, which is the sense in which the original metric was
measuring the wrong thing.

The instruction-level view is also pooled by congruency: whether an instruction's
own term names the side its target actually occupies. Both accounts agree on the
congruent instruction, so the incongruent one is where they part, and pooling
across arrangements doubles the sample for that number since each same-side scene
contributes one incongruent instruction whichever side it was built on.

The comparison is then repeated on the mirrored stimuli, where reflection negates
every recorded side. That is a second measurement of the same quantity on images
the model has not been shown, and it costs nothing beyond predictions the control
factorial already collects.

In [39]:
def report_absolute(result, label):
    if not result.get('n'):
        print(f'--- {label}: no resolved predictions')
        return
    print(f"--- {label}")
    print(f"  n={result['n']}  ({result['n_unresolved']} scenes gave no decidable "
          "prediction and pick no side)")
    print(f"  instructions moving toward their own target: "
          f"{result['agreement']:.1%}")
    print(f"  both instructions correct in the same scene: "
          f"{result['both_correct']:.1%}")
    print('  by configuration:')
    for name, entry in sorted(result['by_configuration'].items()):
        if not entry.get('n'):
            print(f'    {name:16} no resolved predictions')
            continue
        print(f"    {name:16} n={entry['n']:4}  agreement={entry['agreement']:.1%}  "
              f"both correct={entry['both_correct']:.1%}  "
              f"(A={entry['agreement_a']:.1%}, B={entry['agreement_b']:.1%})")
    print('  by congruency of the instruction with its own target:')
    for name, entry in sorted(result.get('by_congruency', {}).items()):
        if not entry.get('n'):
            continue
        print(f"    {name:12} n={entry['n']:4}  agreement={entry['agreement']:.1%}")


if not built.empty:
    absolute = absolute_congruence(built, lateral_sign=IMAGE_X_TO_LATERAL_SIGN)
    if not absolute.get('n'):
        print('target sides were not logged; re-run Notebook 05 so the '
              'constructed scenes carry the recorded target sides')
    else:
        report_absolute(absolute, 'every nonzero prediction')
        report_absolute(
            absolute_congruence(built, lateral_sign=IMAGE_X_TO_LATERAL_SIGN,
                                min_magnitude=BIN_WIDTH),
            f'at least one bin ({BIN_WIDTH:.6f})')
        # Reflection negates every recorded side, so the mirrored stimuli carry
        # the same contrast on images the model has not been shown. Agreement
        # here that matches the baseline is a replication rather than a new
        # result, and a disagreement between the two would point at the
        # convention rather than at the model.
        report_absolute(
            absolute_congruence(built, lateral_sign=IMAGE_X_TO_LATERAL_SIGN,
                                condition='mirror', geometry_sign=-1),
            'replication on the mirrored stimuli')

        print('\nRead: on the same-side configurations, agreement near 100% is '
              'scene grounding and agreement near 50% is a word-to-direction '
              'mapping. Agreement near 0% is scene grounding under an inverted '
              'sign convention rather than a third account, which section 7a '
              'settles. The opposite configuration cannot separate the accounts, '
              'and the incongruent row of the congruency table is where the '
              'separation is concentrated.')

--- every nonzero prediction
  n=240  (0 scenes gave no decidable prediction and pick no side)
  instructions moving toward their own target: 53.5%
  both instructions correct in the same scene: 27.5%
  by configuration:
    opposite         n= 120  agreement=47.5%  both correct=3.3%  (A=76.7%, B=18.3%)
    same_side_left   n=  64  agreement=84.4%  both correct=76.6%  (A=82.8%, B=85.9%)
    same_side_right  n=  56  agreement=31.2%  both correct=23.2%  (A=28.6%, B=33.9%)
  by congruency of the instruction with its own target:
    congruent    n= 360  agreement=51.7%
    incongruent  n= 120  agreement=59.2%
--- at least one bin (0.000225)
  n=224  (16 scenes gave no decidable prediction and pick no side)
  instructions moving toward their own target: 54.0%
  both instructions correct in the same scene: 29.0%
  by configuration:
    opposite         n= 111  agreement=47.7%  both correct=3.6%  (A=78.4%, B=17.1%)
    same_side_left   n=  59  agreement=86.4%  both correct=81.4%  (A=83.1%, B=

## 8. Controls

### 8a. Lexical prior

The same instruction contrast run against a scene it does not describe. If the
difference survives at comparable size, the response is a property of the words
alone and the baseline contrast is not evidence of grounding.

The comparison scene is drawn by a derangement, so no scene ever receives its own
image, and it is a real frame rather than a grey or noise field. An image far
outside the training distribution can drive the model to a constant action, which
would look like an absent lexical prior whether or not one exists.

In [40]:
for source in sorted(usable['scene_source'].unique()):
    frame = usable[usable['scene_source'] == source]
    result = lexical_check(frame)
    print(f'--- {source}')
    for label in ('baseline', 'swapped_scene'):
        entry = result[label]
        if not entry.get('n'):
            print(f'  {label}: not present')
            continue
        print(f"  {label:14} n={entry['n']:4}  mean |difference|={entry['mean_abs']:.6f}  "
              f"median={entry['median']:+.6f}  p={entry['p_value']:.3g}")
    if np.isfinite(result['ratio']):
        print(f"  ratio swapped/baseline = {result['ratio']:.2f}  "
              "(near 1 means the contrast is reproduced without the scene; "
              "near 0 means it depends on the scene)")

--- constructed
  baseline       n= 240  mean |difference|=0.002256  median=+0.000026  p=0.117
  swapped_scene  n= 240  mean |difference|=0.001987  median=+0.000075  p=0.101
  ratio swapped/baseline = 0.88  (near 1 means the contrast is reproduced without the scene; near 0 means it depends on the scene)


### 8b. The term's marginal effect

Each instruction expressed as a deviation from the prediction on the identical
image with the spatial term removed. Measuring against a within-scene reference
removes whatever the scene contributes on its own, which the raw difference
between the two instructions cannot.

If the term carries directional information, the two variants deviate from that
reference in opposite directions and the product of the deviations is negative.

In [41]:
for source in sorted(usable['scene_source'].unique()):
    frame = usable[usable['scene_source'] == source]
    result = term_effect(frame)
    if not result.get('n'):
        print(f'--- {source}: no scenes with both a baseline pair and a neutral '
              'reference')
        continue
    print(f"--- {source}  n={result['n']}")
    print(f"  deviations in opposite directions: {result['opposed_rate']:.1%}")
    print(f"  both deviations exactly zero:      {result['both_zero_rate']:.1%}")
    for label in ('deviation_a', 'deviation_b'):
        entry = result[label]
        print(f"  {label:12} median={entry['median']:+.6f}  p={entry['p_value']:.3g}")
    sep = result['separation']
    print(f"  separation between them: median={sep['median']:+.6f}  "
          f"p={sep['p_value']:.3g}  effect={sep['rank_biserial']:+.2f}")

--- constructed  n=240
  deviations in opposite directions: 28.3%
  both deviations exactly zero:      0.0%
  deviation_a  median=+0.000042  p=0.0305
  deviation_b  median=+0.000041  p=0.311
  separation between them: median=+0.000026  p=0.117  effect=+0.12


## 9. Equivalence testing for nulls

Where a comparison shows no effect, the question is whether the effect is
negligible or merely unproven, and a non-significant p value does not
distinguish them. Each null is therefore tested for equivalence to zero within
one action bin width, a bound set by the measurement rather than by the data: a
difference smaller than one bin cannot change the action the model would execute,
so it is negligible in the only sense that matters here.

An equivalent result is a substantive finding, that the manipulation does not
move the model. A result that is neither significantly different nor
significantly equivalent means the sample cannot resolve the question, which is
worth stating plainly rather than presenting as a null.

The two tests answer different questions, so both can reject at once. A small
but consistent difference in a large sample is detectable while still being
smaller than one action bin: real, and yet too small to change the action the
model would execute. That case is reported as `detectable but sub-bin` rather
than being collapsed into either an effect or a null, since presenting it as
either would misstate what was found.

In [42]:
rows = []
for source in sorted(usable['scene_source'].unique()):
    frame = usable[usable['scene_source'] == source]
    for condition in ('baseline', 'swapped_scene'):
        diffs = paired_difference(frame, condition=condition)['diff']
        if diffs.empty:
            continue
        difference = wilcoxon_paired(diffs)
        equivalence = tost_equivalence(diffs, BIN_WIDTH)
        significant = difference['p_value'] < 0.05
        # The two tests answer different questions and can both reject: a
        # consistent difference far below one bin is real but too small to
        # change the action, which is neither an effect nor a null.
        if significant and equivalence['equivalent']:
            verdict = 'detectable but sub-bin'
        elif significant:
            verdict = 'effect present'
        elif equivalence['equivalent']:
            verdict = 'equivalent to zero'
        else:
            verdict = 'inconclusive'
        rows.append({
            'scene_source': source, 'condition': condition, 'n': difference['n'],
            'median': difference['median'], 'p_difference': difference['p_value'],
            'p_equivalence': equivalence['p_equivalence'], 'verdict': verdict,
        })

if rows:
    print(pd.DataFrame(rows).to_string(index=False))
    print(f'\nequivalence bound: +/- {BIN_WIDTH:.7f} (one action bin)')

scene_source     condition   n   median  p_difference  p_equivalence      verdict
 constructed      baseline 240 0.000026      0.117361       0.837431 inconclusive
 constructed swapped_scene 240 0.000075      0.100568       0.413372 inconclusive

equivalence bound: +/- 0.0002250 (one action bin)


## 10. Manipulation check

A composited object is only useful if the model perceives it. Run with the
term-free instruction, which names the object without locating it, the action
should sometimes point at the pasted instance rather than the original. A rate at
or near zero means the paste is being ignored, and the constructed scenes cannot
support the measurement whichever way the language results came out; the response
would be to escalate the cutout from a box paste to a segmentation mask.

The expectation is one-sided by design. A model free to choose between two
instances need not split evenly, so the check is that the duplicate is chosen a
non-trivial fraction of the time, not that the split is balanced.

In [43]:
# The frozen set, so the check describes the scenes the experiments ran on.
scenes = evaluation_scenes(CONSTRUCTED_DIR)
if scenes:
    neutral = usable[(usable['scene_source'] == 'constructed')
                     & (usable['condition'] == 'neutral')].copy()
    neutral['construct_id'] = neutral['scene_id'].astype(str)
    result = manipulation_rate(neutral, scenes, value_col='c0')
    if result['n']:
        print(f"informative scenes (instances in opposite directions): {result['n']}")
        print(f"action toward the pasted instance: {result['toward_pasted']} "
              f"({result['rate']:.1%})")
        for name, entry in sorted(result['by_configuration'].items()):
            print(f"  {name:16} n={entry['n']:4}  toward pasted={entry['rate']:.1%}")
    else:
        print('no informative neutral predictions on constructed scenes yet')
else:
    print('no frozen evaluation set; screen and freeze in Notebook 03 first')

informative scenes (instances in opposite directions): 120
action toward the pasted instance: 59 (49.2%)
  opposite         n= 120  toward pasted=49.2%
  same_side_left   n=   0  toward pasted=nan%
  same_side_right  n=   0  toward pasted=nan%


## 11. Unaltered Bridge scenes

Held back. Those frames remain in the prediction log for a later validation
analysis after the review in Notebook 04. Mixing an unstratified, mostly
unreviewed pool into this notebook would present a transfer result the labels
do not yet support.

In [44]:
print(f'{n_bridge} unaltered Bridge predictions are in the log and are not '
      'reported here')

8276 unaltered Bridge predictions are in the log and are not reported here


## 12. Summary

Collects the numbers the write-up depends on into one place, so the reported
result and the conditions under which it holds cannot drift apart.

The order matters. If the instrument check fails, the language rows describe a
model that never used the image and should be reported as such. If the
manipulation check fails, the constructed rows describe scenes whose second
instance was invisible. Only when both pass does the same-side contrast answer
the question the study set out to ask.

In [45]:
summary = {}
# Taken from a same-side arrangement, since reflecting an opposite arrangement
# maps the layout onto itself and cannot fail the check.
entry = mirror_built_same_side.get('neutral', {})
if entry.get('n'):
    summary['mirror flip rate (same side, term free)'] = f"{entry['flip_rate']:.1%}"

if not built.empty:
    congruence = congruence_test(built, lateral_sign=IMAGE_X_TO_LATERAL_SIGN)
    summary['paired difference matches geometry'] = f"{congruence['agreement']:.1%}"
    # Paired within the base frame and held to one action bin: the strictest form
    # of the decisive contrast, and the one the write-up quotes.
    decisive = same_side_test(built, min_magnitude=BIN_WIDTH)
    paired = decisive['contrast_paired']
    summary['same-side minus opposite, paired'] = (
        f"{paired['difference']:+.1%} (p={paired['p_value']:.3g}, "
        f"{paired['n_pairs']} frames)")
    summary['same-side minus opposite, unpaired'] = (
        f"{decisive['contrast']['difference']:+.1%} "
        f"(p={decisive['contrast']['p_value']:.3g})")
    absolute = absolute_congruence(built, lateral_sign=IMAGE_X_TO_LATERAL_SIGN,
                                   min_magnitude=BIN_WIDTH)
    for name in ('same_side_left', 'same_side_right'):
        entry = absolute.get('by_configuration', {}).get(name)
        if entry and entry.get('n'):
            summary[f'moves toward own target ({name})'] = (
                f"{entry['agreement']:.1%} (n={entry['n']})")
    incongruent = absolute.get('by_congruency', {}).get('incongruent', {})
    if incongruent.get('n'):
        summary['moves toward own target (incongruent term)'] = (
            f"{incongruent['agreement']:.1%} (n={incongruent['n']})")
    summary['sign convention (IMAGE_X_TO_LATERAL_SIGN)'] = f'{IMAGE_X_TO_LATERAL_SIGN:+d}'

lex = lexical_check(usable)
if np.isfinite(lex['ratio']):
    summary['lexical ratio, swapped over baseline'] = f"{lex['ratio']:.2f}"

continuous_diffs = paired_difference(usable)['diff']
argmax_diffs = paired_difference(usable, continuous=False)['diff']
if len(continuous_diffs) and len(argmax_diffs):
    summary['exact zeros, argmax readout'] = (
        f"{resolution_report(argmax_diffs, BIN_WIDTH)['frac_exact_zero']:.1%}")
    summary['exact zeros, continuous readout'] = (
        f"{resolution_report(continuous_diffs, BIN_WIDTH)['frac_exact_zero']:.1%}")

width = max(len(k) for k in summary) if summary else 0
for key, value in summary.items():
    print(f'{key:<{width}}  {value}')

mirror flip rate (same side, term free)     29.2%
paired difference matches geometry          46.7%
same-side minus opposite, paired            +1.9% (p=0.804, 107 frames)
same-side minus opposite, unpaired          -0.7% (p=1)
moves toward own target (same_side_left)    86.4% (n=59)
moves toward own target (same_side_right)   31.5% (n=54)
moves toward own target (incongruent term)  61.1% (n=113)
sign convention (IMAGE_X_TO_LATERAL_SIGN)   +1
lexical ratio, swapped over baseline        0.88
exact zeros, argmax readout                 40.8%
exact zeros, continuous readout             0.0%
